# Alexinator-240M : pré-entraînement sur une Radeon AI PRO R9700

Ce notebook pré-entraîne notre GPT décodeur (prédiction du token suivant) sur 3 milliards de tokens de français,
avec des réglages prévus pour une **AMD Radeon AI PRO R9700** : RDNA4 (`gfx1201`), 32 Go, bfloat16 matériel.

| | Alexinator-110M (serveur) | Alexinator-240M (ce notebook) |
| --- | --- | --- |
| Blocs / largeur / têtes | 12 / 768 / 12 | 16 / 1 024 / 16 (têtes de 64) |
| Paramètres | 109,5 M | 238,3 M |
| Précision | float32 (Tesla M40) | bfloat16 |
| Micro-batch | fixé à la main | le plus grand qui tient en mémoire, trouvé automatiquement |

C'est la même boucle que `entrainement/pretrain.py` (et que le lab 5), écrite cellule par cellule. Les checkpoints ont
le même format : on peut reprendre un entraînement du notebook avec le script, et inversement, puis passer à la
phase 2 avec `entrainement/sft.py --run alexinator-240m`.

**Avant de lancer :**

1. PyTorch version **ROCm** installé dans l'environnement du notebook (voir la cellule « Carte graphique ») ;
2. pour un essai rapide de toute la chaîne (2 à 3 minutes), mettre `TEST = True` dans la cellule suivante ;
3. le pré-entraînement complet dure environ un jour (estimation) : la cellule « Banc » le mesure en 1 minute.

Une interruption (bouton ■ du notebook) enregistre `dernier.pt` ; relancer la cellule d'entraînement reprend là où on
s'était arrêté.

## 1. Réglages

In [ ]:
TEST = False  # True : mini-modèle sur mini-corpus, pour vérifier toute la chaîne en 2-3 minutes

# Code du projet : le dossier courant s'il contient le dépôt, sinon un clone dans ./alexinator
DEPOT = "https://github.com/axel-lexa/alexinator.git"
BRANCHE = "entrainement/alexinator-110m"
DOSSIER_DEPOT = None  # ou un chemin vers un clone existant

DONNEES = None  # None : entrainement/donnees (corpus de 3 Md de tokens, préparé à l'étape 4 s'il manque)
NOM = "alexinator-240m"  # dossier de sortie dans entrainement/runs/
REPRENDRE = True  # repart de dernier.pt s'il existe

# Modèle : 238 M de paramètres. Têtes de 64 (gérées par les noyaux d'attention AOTriton de RDNA4),
# dimensions multiples de 128 pour les unités matricielles (WMMA).
MODELE = dict(n_layer=16, n_head=16, d_model=1024, d_ff=2816, block_size=1024)

# Entraînement
TOKENS = 3e9  # tokens vus au total (fixe le nombre de pas)
TOKENS_PAR_PAS = 524_288  # 512 séquences de 1 024 tokens par pas d'optimisation
MICRO_BATCH = "auto"  # ou un nombre qui divise TOKENS_PAR_PAS / block_size
LR, LR_MIN, WARMUP = 4e-4, 4e-5, 200  # warmup linéaire puis cosinus
WEIGHT_DECAY, CLIP = 0.1, 1.0
EVAL_TOUS, EVAL_BATCHS, SAUVER_TOUS = 250, 40, 250
COMPILER = True  # torch.compile, avec retour automatique au mode normal s'il échoue
GRAINE = 1337

In [ ]:
if TEST:  # mini-modèle, quelques pas : vérifie la chaîne sans rien abîmer
    MODELE = dict(n_layer=2, n_head=4, d_model=128, d_ff=352, block_size=128)
    TOKENS, TOKENS_PAR_PAS = 16 * 128 * 40, 16 * 128  # 40 pas
    WARMUP, EVAL_TOUS, EVAL_BATCHS, SAUVER_TOUS = 5, 20, 5, 20
    NOM = "test-notebook"

## 2. Code du projet

Le modèle (`alexinator/gpt.py`) et les aides matérielles (`entrainement/materiel.py`) viennent du dépôt.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def trouver_depot():
    if DOSSIER_DEPOT:
        return Path(DOSSIER_DEPOT).resolve()
    for dossier in [Path.cwd(), *Path.cwd().parents]:
        if (dossier / "alexinator" / "gpt.py").exists() and (dossier / "entrainement").exists():
            return dossier
    clone = Path.cwd() / "alexinator"
    if not (clone / ".git").exists():
        subprocess.run(["git", "clone", "--branch", BRANCHE, DEPOT, str(clone)], check=True)
    return clone


ROOT = trouver_depot()
os.chdir(ROOT)
for chemin in (ROOT, ROOT / "entrainement"):
    if str(chemin) not in sys.path:
        sys.path.insert(0, str(chemin))
print("Dépôt :", ROOT)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

## 3. Carte graphique

Sous ROCm, PyTorch garde le nom `cuda` : `torch.cuda` et `device="cuda"` désignent la carte AMD.

Installer PyTorch ROCm (une fois, puis redémarrer le noyau) :

```
pip install torch --index-url https://download.pytorch.org/whl/rocm6.4
```

In [ ]:
import importlib.util

import materiel

if "torch" in sys.modules:
    print("PyTorch était déjà importé : redémarrer le noyau pour que les réglages ROCm s'appliquent.")
materiel.preparer_environnement()  # attention AOTriton sur Radeon ; avant `import torch`

manquants = [m for m in ("tokenizers", "datasets", "numpy", "tqdm", "matplotlib") if importlib.util.find_spec(m) is None]
if manquants:  # jamais torch : pip installerait la version NVIDIA
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *manquants], check=True)

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | {'ROCm ' + torch.version.hip if materiel.est_rocm() else 'CUDA ' + str(torch.version.cuda)}")
print("Carte :", materiel.decrire_carte(device))
if device == "cpu" and not TEST:
    raise SystemExit("Aucun GPU visible : vérifier `rocm-smi` et que PyTorch est la version ROCm.")
if device == "cuda" and not materiel.est_rocm():
    print("Remarque : carte NVIDIA. Le notebook marche, mais les réglages visent une Radeon R9700.")

amp_dtype, scaler = materiel.choisir_precision(device)
compilable, raison = materiel.compilation_possible(device)
print("Précision :", amp_dtype or torch.float32, "| torch.compile :", "oui" if compilable else f"non ({raison})")

## 4. Corpus

Tokenizer BPE de 32 000 tokens, puis Wikipédia FR et FineWeb-2 tokenisés jusqu'à 3 milliards de tokens (`train.bin`, `val.bin`, environ 6 Go). Compter environ une heure ; sauté si le corpus est déjà là.

On peut aussi copier le dossier `entrainement/donnees/` depuis le serveur, où il est déjà prêt : `scp -r serveur:~/alexinator/entrainement/donnees entrainement/`.

In [ ]:
donnees = Path(DONNEES) if DONNEES else ROOT / "entrainement" / ("donnees_test" if TEST else "donnees")
if (donnees / "stats.json").exists():
    print("Corpus prêt :", (donnees / "stats.json").read_text().replace("\n", " "))
else:
    commande = [sys.executable, "-u", "entrainement/preparer_donnees.py", "--sortie", str(donnees)]
    if TEST:
        commande.append("--test")
    with subprocess.Popen(commande, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for ligne in proc.stdout:  # affichage au fil de l'eau
            print(ligne, end="")
    if proc.returncode != 0 or not (donnees / "stats.json").exists():
        raise SystemExit(f"Préparation du corpus en échec (code {proc.returncode}).")

## 5. Modèle

Le micro-batch est choisi en essayant un pas complet (avant, arrière, AdamW) avec 64, 32, 16... séquences, chaque essai dans un processus séparé. On garde la première taille dont le pic reste sous 90 % de la mémoire. L'accumulation s'ajuste pour que le pas reste de `TOKENS_PAR_PAS` tokens.

In [ ]:
import math

import numpy as np
from tokenizers import Tokenizer

from alexinator.gpt import FIN_TEXTE, GPT, GPTConfig

torch.manual_seed(GRAINE)
tokenizer = Tokenizer.from_file(str(donnees / "tokenizer.json"))
train_data = np.memmap(donnees / "train.bin", dtype=np.uint16, mode="r")
val_data = np.memmap(donnees / "val.bin", dtype=np.uint16, mode="r")
cfg = GPTConfig(vocab_size=tokenizer.get_vocab_size(), dropout=0.0, **MODELE)
T = cfg.block_size

sequences = TOKENS_PAR_PAS // T
if MICRO_BATCH == "auto" and device == "cuda":
    candidats = [c for c in (64, 32, 16, 8, 4, 2, 1) if sequences % c == 0]
    micro_batch = materiel.trouver_micro_batch(cfg.to_dict(), amp_dtype, candidats)
else:
    micro_batch = 8 if MICRO_BATCH == "auto" else int(MICRO_BATCH)
assert sequences % micro_batch == 0, "MICRO_BATCH doit diviser TOKENS_PAR_PAS / block_size"
accumulation = sequences // micro_batch
tokens_par_pas = micro_batch * T * accumulation
max_steps = math.ceil(TOKENS / tokens_par_pas)

run_dir = ROOT / "entrainement" / "runs" / NOM
run_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = run_dir / "dernier.pt"
model = GPT(cfg).to(device)
optimizer = model.configure_optimizer(LR, WEIGHT_DECAY)
start_step, best_val = 0, float("inf")
if REPRENDRE and ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step, best_val = ckpt["step"], ckpt.get("best_val", float("inf"))
    del ckpt
    print(f"Reprise au pas {start_step}")
raw_model = model  # le modèle compilé partage ses poids avec raw_model (sauvegarde, génération)

print(f"Modèle : {model.num_params() / 1e6:.1f} M de paramètres")
print(f"Corpus : {len(train_data) / 1e9:.3f} Md tokens (train), {len(val_data) / 1e6:.1f} M (val)")
print(f"Micro-batch {micro_batch} x accumulation {accumulation} x {T} tokens = {tokens_par_pas:,} tokens par pas")
print(f"{max_steps} pas = {max_steps * tokens_par_pas / 1e9:.2f} Md de tokens")

## 6. Fonctions de la boucle

In [ ]:
import csv
import json
import time
from contextlib import nullcontext

autocast = (lambda: torch.autocast("cuda", dtype=amp_dtype)) if amp_dtype else nullcontext


def get_batch(split):
    """Comme au lab 5 : fenêtres prises au hasard ; la cible est la fenêtre décalée d'un token."""
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - T - 1, (micro_batch,))
    x = torch.stack([torch.from_numpy(data[i : i + T].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + T].astype(np.int64)) for i in ix])
    if device == "cuda":
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y


def lr_at(step):
    """Warmup linéaire, puis décroissance en cosinus jusqu'à LR_MIN."""
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = min(1.0, (step - WARMUP) / max(1, max_steps - WARMUP))
    return LR_MIN + 0.5 * (LR - LR_MIN) * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(EVAL_BATCHS):
            x, y = get_batch(split)
            with autocast():
                losses.append(model(x, y)[1].item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out


def samples(prompts=("Le film", "En 1997,", "Le réalisateur"), length=60):
    eot = tokenizer.token_to_id(FIN_TEXTE)
    texts = []
    for prompt in prompts:
        idx = torch.tensor([tokenizer.encode(prompt).ids], device=device)
        with autocast():
            out = raw_model.generate(idx, length, temperature=0.8, top_k=50, stop_id=eot)
        texts.append(tokenizer.decode(out[0].tolist()))
    return texts


def save(step, path):
    """Même format que entrainement/pretrain.py : lisible par sft.py et par le jeu."""
    torch.save(
        {
            "model": raw_model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "config": cfg.to_dict(),
            "step": step,
            "best_val": best_val,
            "tokenizer": str(donnees / "tokenizer.json"),
        },
        path,
    )


def train_step(step):
    """Un pas d'optimisation = `accumulation` micro-batchs (gradients additionnés). Renvoie la loss."""
    lr = lr_at(step)
    for group in optimizer.param_groups:
        group["lr"] = lr
    for _ in range(accumulation):
        x, y = get_batch("train")
        with autocast():
            _, loss = model(x, y)
        loss = loss / accumulation
        (scaler.scale(loss) if scaler else loss).backward()
    if scaler:
        scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), CLIP)
    if scaler:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    return loss.item() * accumulation, lr

## 7. Compilation

`torch.compile` ne compile qu'au premier appel : une erreur (Triton absent, carte non gérée) n'apparaîtrait qu'au premier pas. On fait donc un passage d'essai ici, et on garde le mode normal s'il échoue.

In [ ]:
model = raw_model
if COMPILER and compilable:
    print("Compilation (1 à 3 minutes)...")
    try:
        model = torch.compile(raw_model)
        x, y = get_batch("train")
        with autocast():
            _, loss = model(x, y)
        loss.backward()
        print("Compilation réussie.")
    except Exception as error:
        print(f"torch.compile indisponible ({type(error).__name__}: {str(error)[:300]}) : mode normal.")
        model = raw_model
    finally:
        raw_model.zero_grad(set_to_none=True)
else:
    print("Sans compilation.")

## 8. Banc (facultatif)

Mesure la vitesse sur quelques pas **sans modifier le modèle** (gradients calculés puis jetés) et estime la durée totale. Mettre `BANC = 0` pour sauter cette étape.

In [ ]:
BANC = 0 if TEST else 8  # pas mesurés

if BANC:
    model.train()

    def pas_sans_mise_a_jour():
        for _ in range(accumulation):
            x, y = get_batch("train")
            with autocast():
                _, loss = model(x, y)
            (loss / accumulation).backward()
        optimizer.zero_grad(set_to_none=True)

    pas_sans_mise_a_jour()  # chauffe (caches, noyaux)
    if device == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(BANC):
        pas_sans_mise_a_jour()
    if device == "cuda":
        torch.cuda.synchronize()
    vitesse = BANC * tokens_par_pas / (time.time() - t0)
    memoire = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
    restant = (max_steps - start_step) * tokens_par_pas / vitesse / 3600
    print(f"{vitesse / 1e3:.1f} k tokens/s | mémoire max {memoire:.1f} Go | reste ~{restant:.1f} h pour {TOKENS / 1e9:.1f} Md de tokens")

## 9. Entraînement

Barre de progression, évaluation et échantillons de texte tous les `EVAL_TOUS` pas, sauvegarde tous les `SAUVER_TOUS` pas (`dernier.pt`) et à chaque meilleure loss de validation (`meilleur.pt`). Journal dans `journal.csv`, échantillons dans `echantillons.txt`. Le bouton ■ interrompt proprement (sauvegarde puis arrêt).

In [ ]:
from tqdm.auto import tqdm

log_path = run_dir / "journal.csv"
new_log = not log_path.exists() or start_step == 0
log_file = open(log_path, "w" if new_log else "a", newline="")
log = csv.writer(log_file)
if new_log:
    log.writerow(["pas", "tokens_vus", "loss_train", "loss_val", "lr", "tokens_par_s", "memoire_go"])
(run_dir / "config.json").write_text(json.dumps({**cfg.to_dict(), "micro_batch": micro_batch,
                                                 "accumulation": accumulation, "tokens": TOKENS, "lr": LR}, indent=2))

model.train()
barre = tqdm(total=max_steps, initial=start_step, unit="pas", dynamic_ncols=True)
step = start_step
t0, tokens_depuis = time.time(), 0
try:
    for step in range(start_step, max_steps):
        if step % EVAL_TOUS == 0 or step == max_steps - 1:
            losses = evaluate()
            mem = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
            barre.write(f"[pas {step}/{max_steps}] loss train {losses['train']:.3f} | val {losses['val']:.3f} | "
                        f"lr {lr_at(step):.2e} | mémoire max {mem:.1f} Go")
            log.writerow([step, step * tokens_par_pas, f"{losses['train']:.4f}", f"{losses['val']:.4f}",
                          f"{lr_at(step):.3e}", "", f"{mem:.2f}"])
            log_file.flush()
            textes = samples()
            with open(run_dir / "echantillons.txt", "a", encoding="utf-8") as f:
                f.write(f"\n=== pas {step} (loss val {losses['val']:.3f}) ===\n" + "\n---\n".join(textes) + "\n")
            barre.write("   « " + textes[0].replace("\n", " ")[:150] + " »")
            if losses["val"] < best_val and step > start_step:
                best_val = losses["val"]
                save(step, run_dir / "meilleur.pt")
        if step > start_step and step % SAUVER_TOUS == 0:
            save(step, ckpt_path)

        loss, lr = train_step(step)
        tokens_depuis += tokens_par_pas
        barre.update(1)
        if step % 10 == 0:
            if device == "cuda":
                torch.cuda.synchronize()
            vitesse = tokens_depuis / (time.time() - t0)
            t0, tokens_depuis = time.time(), 0
            barre.set_postfix(loss=f"{loss:.3f}", lr=f"{lr:.1e}", tok_s=f"{vitesse / 1e3:.1f}k")
            log.writerow([step, step * tokens_par_pas, f"{loss:.4f}", "", f"{lr:.3e}", f"{vitesse:.0f}", ""])
            log_file.flush()
    step = max_steps
    save(step, ckpt_path)
    if best_val == float("inf"):  # entraînement très court : pas encore de « meilleur » modèle
        save(step, run_dir / "meilleur.pt")
    print(f"Terminé. Modèles dans {run_dir} (meilleur.pt = plus petite loss de validation).")
except KeyboardInterrupt:
    save(step, ckpt_path)
    print(f"Interrompu au pas {step} : dernier.pt enregistré. Relancer cette cellule pour reprendre.")
    start_step = step
finally:
    barre.close()
    log_file.close()

## 10. Courbes

In [ ]:
import matplotlib.pyplot as plt

lignes = list(csv.DictReader(open(log_path, encoding="utf-8")))
pas_train = [(int(r["pas"]), float(r["loss_train"])) for r in lignes if r["tokens_par_s"]]
pas_val = [(int(r["pas"]), float(r["loss_val"])) for r in lignes if r["loss_val"]]
fig, ax = plt.subplots(figsize=(8, 4))
if pas_train:
    ax.plot(*zip(*pas_train), label="train (pas courant)", alpha=0.6)
if pas_val:
    ax.plot(*zip(*pas_val), "o-", label="validation")
ax.set_xlabel("pas")
ax.set_ylabel("loss")
ax.set_title(f"{NOM} : {raw_model.num_params() / 1e6:.0f} M de paramètres")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 11. Génération

In [ ]:
for texte in samples(("Le film", "Le gardien", "En 1997,", "Le réalisateur"), length=80):
    print(texte.replace("\n", " "), "\n---")

## Suite

- **Reprendre** après une coupure : relancer le notebook (avec `REPRENDRE = True`), ou en ligne de commande
  `python entrainement/pretrain.py --profil r9700 --reprendre`.
- **Phase 2** (spécialisation en gardien), quand `finetune/data/train.jsonl` est prêt :
  `python entrainement/sft.py --run alexinator-240m`.
- **Jouer** contre le modèle : voir la section « Jouer contre notre modèle » de `docs/entrainement.md`.